<a href="https://colab.research.google.com/github/Nihat0311/NLP/blob/main/Data_Converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets transformers[torch] accelerate evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=4716d021197c825fdb95afb51aab16f1496be22053d54a96d79c3cf2657b6bda
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
import re
from datasets import load_dataset
from datetime import datetime

dataset = load_dataset("lilas12/date-converter-100k")

def format_target_date(example):
    original_target = example['target'].strip()
    try:
        date_obj = datetime.strptime(original_target, "%Y-%m-%d")
        example['target_formatted'] = date_obj.strftime("%d.%m.%Y")
    except ValueError:
        parts = re.split(r'[-/\s]+', original_target)
        if len(parts) == 3:
            year = parts[0].zfill(4)
            month = parts[1].zfill(2)
            day = parts[2].zfill(2)
            example['target_formatted'] = f"{day}.{month}.{year}"
        else:
            example['target_formatted'] = original_target
    return example

dataset = dataset.map(format_target_date)


print("Nümunə Giriş (Source):", dataset['train'][0]['source'])
print("Nümunə Çıxış (Target Formatted):", dataset['train'][0]['target_formatted'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/815 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.02M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/387k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/386k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/80000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Nümunə Giriş (Source): 9156 may 13
Nümunə Çıxış (Target Formatted): 13.05.9156


In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

prefix = "convert date: "

def preprocess_function(examples):

    inputs = [prefix + text for text in examples["source"]]
    targets = [text for text in examples["target_formatted"]]

    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(text_target=targets, max_length=32, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Map:   0%|          | 0/80000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Modelimizin işləyəcəyi cihaz: {device}")

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
model.to(device)

Modelimizin işləyəcəyi cihaz: cuda


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./date_converter_model",
    eval_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)


small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(20000))
small_eval_dataset = tokenized_datasets["validation"].shuffle(seed=42).select(range(2000))

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.489747,0.189304


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=625, training_loss=0.44846377563476564, metrics={'train_runtime': 76.8796, 'train_samples_per_second': 260.147, 'train_steps_per_second': 8.13, 'total_flos': 98317359906816.0, 'train_loss': 0.44846377563476564, 'epoch': 1.0})

In [ ]:
def convert_date(text_input):
    input_text = "convert date: " + text_input

    inputs = tokenizer(input_text, return_tensors="pt").to(device)


    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=32)

    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return output_text


print("Giriş: '9156 may 13' -> Çıxış:", convert_date("9156 may 13"))
print("Giriş: 'the 18th of november 9508' -> Çıxış:", convert_date("the 18th of november 9508"))


print("Giriş: 'july 4, 2026' -> Çıxış:", convert_date("july 4, 2026"))
print("Giriş: 'on monday may 4 3744' -> Çıxış:", convert_date("on monday may 4 3744"))

Giriş: '9156 may 13' -> Çıxış: 13.05.9156
Giriş: 'the 18th of november 9508' -> Çıxış: 18.11.9508
Giriş: 'july 4, 2026' -> Çıxış: 04.07.2026
Giriş: 'on monday may 4 3744' -> Çıxış: 04.05.3744
